# Từ nghiệm giải tích đến FEM và PINN: thực hành trên Colab

Một notebook **tự chứa mã**, không cần tải file phụ. Chạy tuần tự trên CPU bằng Runtime → Run all, sau đó lưu bản sao vào Google Drive để làm bài tập. Các ô mã dưới đây được chép từ script 03 và 04 của dự án để nhóm có thể xem và sửa trực tiếp.

Bài toán không thứ nguyên: $u^{(4)}(x)=1$, $0<x<1$; $u(0)=u(1)=u''(0)=u''(1)=0$. Quy ước chuyển vị dương xuống, mômen $m=-u''$. Nghiệm giải tích là **ground truth chỉ để kiểm tra**, không đi vào loss PINN.

## 0. Kiểm tra môi trường
Colab thông thường có sẵn NumPy, SciPy, Matplotlib và PyTorch. Chọn CPU; ghi phiên bản các thư viện trong báo cáo.

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
print("Python", sys.version.split()[0], "NumPy", np.__version__, "PyTorch", torch.__version__)

## 1. Nghiệm giải tích
Tích phân bốn lần và áp điều kiện biên cho $u_*(x)=(x^4-2x^3+x)/24$ và $m_*(x)=x(1-x)/2$. Tự lấy đạo hàm để kiểm tra phương trình và bốn điều kiện biên.

In [ ]:
def exact_u(x):
    x = np.asarray(x)
    return (x**4 - 2*x**3 + x)/24

def exact_m(x):
    x = np.asarray(x)
    return x*(1-x)/2

assert np.isclose(exact_u(0), 0) and np.isclose(exact_u(1), 0)
print("u*(1/2) =", exact_u(0.5), "; m*(1/2) =", exact_m(0.5))

**Bài tập A.** Chứng minh $u_*^{(4)}=1$ và $u_*''(0)=u_*''(1)=0$. Vì sao kiểm tra mỗi chuyển vị ở hai gối là chưa đủ?

## 2. FEM Hermite: dạng yếu đến nghiệm
Tìm $u\in V_0=\{v\in H^2(0,1):v(0)=v(1)=0\}$ sao cho
$\int_0^1 u''v''\,dx=\int_0^1v\,dx$ với mọi $v\in V_0$.
Phần tử Hermite bậc ba có hai bậc tự do ở mỗi nút: chuyển vị và góc xoay. Chạy ô dưới để định nghĩa ma trận phần tử, lắp ráp và đánh giá nghiệm trong phần tử. Đối chiếu [dẫn xuất FEM](https://nhannguyenminh.github.io/intro2pinns/beam-fem-foundations.html).

In [ ]:
"""Bài 03: giải hệ tuyến tính và FEM Hermite cho dầm tựa đơn.

Chạy trong CMD: python 03-linear-fem.py
Nghiệm được đánh giá bên trong phần tử, không chỉ tại các nút.
"""

import numpy as np
from scipy.integrate import trapezoid
from scipy.linalg import solve


def element_matrices(h: float, bending_stiffness: float = 1.0, load: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Ma trận phần tử với thứ tự [w_i, theta_i, w_j, theta_j], theta=w'."""
    k = (bending_stiffness / h**3) * np.array(
        [
            [12, 6 * h, -12, 6 * h],
            [6 * h, 4 * h**2, -6 * h, 2 * h**2],
            [-12, -6 * h, 12, -6 * h],
            [6 * h, 2 * h**2, -6 * h, 4 * h**2],
        ],
        dtype=float,
    )
    f = (load * h / 12) * np.array([6, h, 6, -h], dtype=float)
    return k, f


def assemble_beam(elements: int, length: float = 1.0, bending_stiffness: float = 1.0, load: float = 1.0) -> np.ndarray:
    """Lắp ráp và giải hệ; trả về [w_0,theta_0,...,w_n,theta_n]."""
    if elements < 1 or length <= 0 or bending_stiffness <= 0:
        raise ValueError("elements >= 1, length > 0 and EI > 0 are required")
    h = length / elements
    ndof = 2 * (elements + 1)
    stiffness = np.zeros((ndof, ndof))
    force = np.zeros(ndof)
    k_e, f_e = element_matrices(h, bending_stiffness, load)

    for element in range(elements):
        dofs = np.arange(2 * element, 2 * element + 4)
        stiffness[np.ix_(dofs, dofs)] += k_e
        force[dofs] += f_e

    # Dầm tựa đơn: chỉ w ở hai đầu bằng 0; góc xoay là bậc tự do tự do.
    fixed = np.array([0, 2 * elements])
    free = np.setdiff1d(np.arange(ndof), fixed)
    solution = np.zeros(ndof)
    solution[free] = solve(stiffness[np.ix_(free, free)], force[free], assume_a="pos")
    assert np.allclose(stiffness, stiffness.T)
    return solution


def evaluate_beam(points: np.ndarray, solution: np.ndarray, length: float = 1.0, bending_stiffness: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Nội suy w_h và M_h=-EI*w_h'' tại các điểm trong [0,L]."""
    elements = len(solution) // 2 - 1
    h = length / elements
    points = np.asarray(points, dtype=float)
    if np.any((points < 0) | (points > length)):
        raise ValueError("Evaluation points must be in [0,L]")
    element_ids = np.minimum((points / h).astype(int), elements - 1)
    s = (points - element_ids * h) / h
    n = np.stack(
        (1 - 3 * s**2 + 2 * s**3, h * (s - 2 * s**2 + s**3),
         3 * s**2 - 2 * s**3, h * (-s**2 + s**3)),
        axis=1,
    )
    n_xx = np.stack(
        ((-6 + 12 * s) / h**2, (-4 + 6 * s) / h,
         (6 - 12 * s) / h**2, (-2 + 6 * s) / h),
        axis=1,
    )
    dofs = 2 * element_ids[:, None] + np.arange(4)
    local_solution = solution[dofs]
    displacement = np.sum(n * local_solution, axis=1)
    moment = -bending_stiffness * np.sum(n_xx * local_solution, axis=1)
    return displacement, moment


def reference_fields(points: np.ndarray, length: float = 1.0, bending_stiffness: float = 1.0, load: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    w = load * (points**4 - 2 * length * points**3 + length**3 * points) / (24 * bending_stiffness)
    moment = load * points * (length - points) / 2
    return w, moment


def relative_l2(numerical: np.ndarray, reference: np.ndarray, points: np.ndarray) -> float:
    return float(np.sqrt(trapezoid((numerical - reference) ** 2, points) / trapezoid(reference**2, points)))


def main() -> None:
    a = np.array([[3.0, 1.0], [1.0, 2.0]])
    b = np.array([9.0, 8.0])
    x = solve(a, b)
    print("System A @ x = b: x =", x, "; residual =", np.linalg.norm(a @ x - b))
    print(f"{'elements':>8} {'error w':>14} {'error M':>14}")
    for elements in (1, 2, 4, 8):
        solution = assemble_beam(elements)
        # Tránh trùng nút FEM khi đánh giá mômen, vốn có thể nhận hai giá trị phía trái/phải.
        points = (np.arange(2000) + 0.5) / 2000
        w_h, m_h = evaluate_beam(points, solution)
        w_star, m_star = reference_fields(points)
        print(f"{elements:8d} {relative_l2(w_h,w_star,points):14.6e} {relative_l2(m_h,m_star,points):14.6e}")
    assert np.allclose(assemble_beam(2)[[0, 4]], 0)

In [ ]:
K1, f1 = element_matrices(1.0)
d1 = assemble_beam(1)
print("K phần tử:", K1, sep="\n")
print("f phần tử:", f1)
print("Nghiệm 1 phần tử [u0, theta0, u1, theta1]:", d1)
assert np.allclose(K1, K1.T)
assert np.allclose(d1, [0, 1/24, 0, -1/24])

Lưới đánh giá dùng trung điểm của 800 ô nhỏ, tránh nút FEM nơi mômen có thể nhận giá trị trái/phải khác nhau. Sai số tương đối dùng cùng một lưới cho FEM và PINN.

In [ ]:
x_eval = (np.arange(800)+0.5)/800
u_star, m_star = exact_u(x_eval), exact_m(x_eval)
print(f"{'N':>4} {'E_u':>13} {'E_m':>13} {'time (s)':>12}")
fem_rows = []
for n in (1, 2, 4, 8):
    t0 = time.perf_counter()
    d = assemble_beam(n)
    fem_time = time.perf_counter()-t0
    u_fem, m_fem = evaluate_beam(x_eval, d)
    eu = relative_l2(u_fem, u_star, x_eval)
    em = relative_l2(m_fem, m_star, x_eval)
    fem_rows.append((n, eu, em, fem_time))
    print(f"{n:4d} {eu:13.4e} {em:13.4e} {fem_time:12.4e}")
d8 = assemble_beam(8)
u_fem, m_fem = evaluate_beam(x_eval, d8)

**Bài tập B.** Thêm $N=16$. Nêu vì sao $K_e$ riêng lẻ suy biến nhưng hệ đã áp gối tựa giải được. Giải thích vì sao cần đánh giá nghiệm **bên trong phần tử**, không chỉ ở các nút.

## 3. PINN: mạng, đạo hàm tự động, loss
Mạng $u_\theta(x)$ dự đoán chuyển vị. Tanh trơn để lấy đạo hàm bậc bốn. Loss phương trình là trung bình bình phương $u_\theta^{(4)}(x_c)-1$; loss biên là tổng bình phương $u_\theta(0),u_\theta(1),u_\theta''(0),u_\theta''(1)$. Dùng $L=L_{\rm PDE}+\lambda_{\rm BC}L_{\rm BC}$; nghiệm giải tích **không tham gia huấn luyện**. Xem [dẫn xuất PINN](https://nhannguyenminh.github.io/intro2pinns/beam-pinn-foundations.html).

**Trước khi chạy ô PINN dài bên dưới:** hoàn thành [notebook 04 từng bước](https://colab.research.google.com/github/NhanNguyenMinh/intro2pinns/blob/main/learn-python/04-torch-pinn.ipynb). Bạn cần chỉ được trong mã: `requires_grad=True` cho tọa độ, `create_graph=True` để lấy đạo hàm cao, residual miền, bốn điều kiện biên, và chu trình `zero_grad → backward → step`. Notebook 05 dùng lại cùng ý tưởng để đặt FEM và PINN trên một lưới kiểm chứng.


In [ ]:
"""Bài 04: đạo hàm tự động và một PINN tối thiểu cho dầm tựa đơn.

Chạy trong CMD: python 04-torch-pinn.py --steps 200
Đây là baseline học tập, không phải kết luận cuối về độ chính xác của PINN.
"""

import argparse
import csv
from pathlib import Path

import torch
from torch import nn


def derivative(output: torch.Tensor, coordinate: torch.Tensor, order: int) -> torch.Tensor:
    """Đạo hàm theo tọa độ; mỗi đầu ra chỉ phụ thuộc vào điểm đầu vào cùng hàng."""
    result = output
    for _ in range(order):
        result = torch.autograd.grad(
            result,
            coordinate,
            grad_outputs=torch.ones_like(result),
            create_graph=True,
        )[0]
    return result


def verify_autograd() -> None:
    x = torch.linspace(0, 1, 5, dtype=torch.float64).reshape(-1, 1).requires_grad_(True)
    polynomial = x**4 - 2 * x**3 + x
    fourth = derivative(polynomial, x, 4)
    assert torch.allclose(fourth, torch.full_like(fourth, 24.0), atol=1e-12)
    print("Autograd: fourth derivative of x^4-2*x^3+x is 24 (passed).")


def make_model() -> nn.Module:
    """Tanh đủ trơn để residual dạng mạnh dùng đạo hàm bậc bốn."""
    return nn.Sequential(
        nn.Linear(1, 32),
        nn.Tanh(),
        nn.Linear(32, 32),
        nn.Tanh(),
        nn.Linear(32, 1),
    ).double()


def train_pinn(steps: int = 200, n_collocation: int = 32, boundary_weight: float = 10.0, seed: int = 7) -> tuple[nn.Module, list[float]]:
    if steps < 1 or n_collocation < 2 or boundary_weight <= 0:
        raise ValueError("steps >= 1, n_collocation >= 2 and boundary_weight > 0 are required")
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    model = make_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    collocation = ((torch.arange(n_collocation, dtype=torch.float64) + 0.5) / n_collocation).reshape(-1, 1).requires_grad_(True)
    boundary = torch.tensor([[0.0], [1.0]], dtype=torch.float64, requires_grad=True)
    history = []

    for _ in range(steps):
        optimizer.zero_grad()
        u_c = model(collocation)
        residual = derivative(u_c, collocation, 4) - 1.0
        loss_pde = torch.mean(residual**2)

        u_b = model(boundary)
        u_b_xx = derivative(u_b, boundary, 2)
        loss_boundary = torch.sum(u_b**2 + u_b_xx**2)
        loss = loss_pde + boundary_weight * loss_boundary
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))

        # Tọa độ là lá của đồ thị autograd; không dùng gradient theo tọa độ để tối ưu.
        collocation.grad = None
        boundary.grad = None
    return model, history


def evaluate(model: nn.Module) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, float, float]:
    coordinate = torch.linspace(0, 1, 401, dtype=torch.float64).reshape(-1, 1).requires_grad_(True)
    predicted = model(coordinate)
    predicted_moment = -derivative(predicted, coordinate, 2)
    exact = (coordinate**4 - 2 * coordinate**3 + coordinate) / 24
    exact_moment = coordinate * (1 - coordinate) / 2
    x = coordinate[:, 0]
    error_u = torch.sqrt(torch.trapz((predicted[:, 0] - exact[:, 0]) ** 2, x) / torch.trapz(exact[:, 0] ** 2, x))
    error_m = torch.sqrt(torch.trapz((predicted_moment[:, 0] - exact_moment[:, 0]) ** 2, x) / torch.trapz(exact_moment[:, 0] ** 2, x))
    return (coordinate.detach(), predicted.detach(), exact.detach(),
            predicted_moment.detach(), exact_moment.detach(),
            float(error_u.detach()), float(error_m.detach()))


def main() -> None:
    parser = argparse.ArgumentParser(description="Minimal PINN for normalized beam u''''=1")
    parser.add_argument("--steps", type=int, default=200, help="number of Adam steps")
    parser.add_argument("--seed", type=int, default=7)
    args = parser.parse_args()

    verify_autograd()
    model, history = train_pinn(steps=args.steps, seed=args.seed)
    x, predicted, exact, predicted_moment, exact_moment, error_u, error_m = evaluate(model)
    print(f"PyTorch {torch.__version__}; seed={args.seed}; steps={args.steps}")
    print(f"loss: {history[0]:.4e} -> {history[-1]:.4e}")
    print(f"Relative errors: u={error_u:.4e}, M={error_m:.4e}")

    output_dir = Path(__file__).resolve().parent / "outputs"
    output_dir.mkdir(exist_ok=True)
    curve_path = output_dir / "pinn_curve.csv"
    loss_path = output_dir / "pinn_loss.csv"
    with curve_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["xi", "u_pinn", "u_exact", "M_pinn", "M_exact"])
        writer.writerows(zip(x[:, 0].tolist(), predicted[:, 0].tolist(),
                             exact[:, 0].tolist(), predicted_moment[:, 0].tolist(),
                             exact_moment[:, 0].tolist()))
    with loss_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["step", "loss"])
        writer.writerows((step, value) for step, value in enumerate(history, start=1))
    print(f"Saved data: {curve_path} and {loss_path}")

In [ ]:
verify_autograd()
pinn_t0 = time.perf_counter()
model, history = train_pinn(steps=200, n_collocation=32, boundary_weight=10.0, seed=7)
pinn_time = time.perf_counter() - pinn_t0
x_t, u_t, exact_t, m_t, exact_m_t, eu_pinn, em_pinn = evaluate(model)
print(f"Loss đầu/cuối: {history[0]:.3e} -> {history[-1]:.3e}")
print(f"PINN: E_u={eu_pinn:.3e}, E_m={em_pinn:.3e}, training={pinn_time:.2f} s")

## 4. So sánh trên cùng lưới
Tính lại sai số PINN trên chính 800 điểm đã dùng cho FEM. Vẽ nghiệm giải tích, FEM 8 phần tử và PINN. Loss nhỏ **không tự bảo đảm** nghiệm hoặc mômen chính xác.

In [ ]:
x_grid = torch.tensor(x_eval.reshape(-1, 1), dtype=torch.float64, requires_grad=True)
u_grid = model(x_grid)
m_grid = -derivative(u_grid, x_grid, 2)
u_pinn = u_grid.detach().numpy().ravel()
m_pinn = m_grid.detach().numpy().ravel()
print(f"{'Phương pháp':<16} {'E_u':>12} {'E_m':>12} {'time (s)':>12}")
print(f"{'FEM N=8':<16} {relative_l2(u_fem,u_star,x_eval):12.4e} {relative_l2(m_fem,m_star,x_eval):12.4e} {fem_rows[-1][3]:12.4e}")
print(f"{'PINN':<16} {relative_l2(u_pinn,u_star,x_eval):12.4e} {relative_l2(m_pinn,m_star,x_eval):12.4e} {pinn_time:12.4e}")
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for panel, truth, fem, pinn, title in [
    (ax[0], u_star, u_fem, u_pinn, "Chuyển vị"),
    (ax[1], m_star, m_fem, m_pinn, "Mômen"),
]:
    panel.plot(x_eval, truth, "k-", label="Giải tích")
    panel.plot(x_eval, fem, "--", label="FEM 8")
    panel.plot(x_eval, pinn, ":", label="PINN")
    panel.set(xlabel="x", title=title)
    panel.grid(alpha=.3); panel.legend()
ax[2].semilogy(history)
ax[2].set(xlabel="Bước Adam", ylabel="Tổng loss", title="Huấn luyện PINN")
ax[2].grid(alpha=.3)
plt.tight_layout()
plt.show()

## 5. Bài tập nhóm và sản phẩm nộp
1. **FEM:** lập bảng $N=1,2,4,8,16$ và hai sai số $E_u,E_m$.
2. **PINN:** giữ kiến trúc và optimizer; thay **một** biến: số điểm collocation $16,32,64$ **hoặc** trọng số biên $1,10,100$. Chạy ít nhất 3 seed/cấu hình.
3. **Kiểm chứng:** đo bốn điều kiện biên của PINN và giải thích quan hệ giữa loss, sai số chuyển vị và sai số mômen.
4. **Nộp:** notebook chạy lại từ đầu, bảng cấu hình/seed/sai số/thời gian, hai hình, phiên bản thư viện và báo cáo ngắn Word hoặc LaTeX.

Thời gian FEM và thời gian PINN cần được đo cùng máy, lặp lại và trình bày rõ cách đo. Ví dụ này là điểm khởi đầu học tập, chưa phải kết luận về hiệu năng của hai phương pháp.